In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

In [2]:
train_labels = pd.read_csv("train.csv")
train_localizers = pd.read_csv("train_localizers.csv")

In [3]:
train_labels.head()

,SeriesInstanceUID,PatientAge,PatientSex,Modality,Left Infraclinoid Internal Carotid Artery,Right Infraclinoid Internal Carotid Artery,Left Supraclinoid Internal Carotid Artery,Right Supraclinoid Internal Carotid Artery,Left Middle Cerebral Artery,Right Middle Cerebral Artery,Anterior Communicating Artery,Left Anterior Cerebral Artery,Right Anterior Cerebral Artery,Left Posterior Communicating Artery,Right Posterior Communicating Artery,Basilar Tip,Other Posterior Circulation,Aneurysm Present
0,1.2.826.0.1.3680043.8.498.10004044428023505108...,64,Female,MRA,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1.2.826.0.1.3680043.8.498.10004684224894397679...,76,Female,MRA,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,1.2.826.0.1.3680043.8.498.10005158603912009425...,58,Male,CTA,0,0,0,0,0,0,0,0,0,0,0,0,1,1
3,1.2.826.0.1.3680043.8.498.10009383108068795488...,71,Male,MRA,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,1.2.826.0.1.3680043.8.498.10012790035410518400...,48,Female,MRA,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [4]:
train_labels.columns

Index(['SeriesInstanceUID', 'PatientAge', 'PatientSex', 'Modality',
       'Left Infraclinoid Internal Carotid Artery',
       'Right Infraclinoid Internal Carotid Artery',
       'Left Supraclinoid Internal Carotid Artery',
       'Right Supraclinoid Internal Carotid Artery',
       'Left Middle Cerebral Artery', 'Right Middle Cerebral Artery',
       'Anterior Communicating Artery', 'Left Anterior Cerebral Artery',
       'Right Anterior Cerebral Artery', 'Left Posterior Communicating Artery',
       'Right Posterior Communicating Artery', 'Basilar Tip',
       'Other Posterior Circulation', 'Aneurysm Present'],
      dtype='object')

In [5]:
type(train_labels)

pandas.core.frame.DataFrame

In [6]:
series_with = train_labels[train_labels["Aneurysm Present"].eq(1)]["SeriesInstanceUID"]
train_labels[train_labels["Aneurysm Present"].eq(1)]["SeriesInstanceUID"].iloc[0:10]

2     1.2.826.0.1.3680043.8.498.10005158603912009425...
8     1.2.826.0.1.3680043.8.498.10022796280698534221...
9     1.2.826.0.1.3680043.8.498.10023411164590664678...
10    1.2.826.0.1.3680043.8.498.10030095840917973694...
12    1.2.826.0.1.3680043.8.498.10034081836061566510...
13    1.2.826.0.1.3680043.8.498.10035643165968342618...
17    1.2.826.0.1.3680043.8.498.10042423585566957032...
18    1.2.826.0.1.3680043.8.498.10042474696169267476...
23    1.2.826.0.1.3680043.8.498.10058383541003792190...
25    1.2.826.0.1.3680043.8.498.10063454172499468887...
Name: SeriesInstanceUID, dtype: object

In [14]:
from pathlib import Path

series_root = Path("/Volumes/X9 Pro/aneurysm/series")
uid = series_with.iloc[1]
#uid = '1.2.826.0.1.3680043.8.498.99892390884723813599532075083872271516' this one is the one that has 4 aneurysm
p = series_root/uid
if p.is_dir():
    print("Found: ", p)
else:
    print("Not found: ", p)

Found:  /Volumes/X9 Pro/aneurysm/series/1.2.826.0.1.3680043.8.498.99892390884723813599532075083872271516


In [15]:
import shutil
desktop = Path.home() / "Desktop"      
dst = desktop/"tochange4aneurysm"
shutil.copytree(p, dst, dirs_exist_ok=True)
print("Copied to: ", dst)

Copied to:  /Users/nicolas/Desktop/tochange4aneurysm


## Find the corresponding center of the aneurysm

In [9]:
train_localizers.head()

,SeriesInstanceUID,SOPInstanceUID,coordinates,location
0,1.2.826.0.1.3680043.8.498.10005158603912009425...,1.2.826.0.1.3680043.8.498.10775329348174902199...,"{'x': 258.3621186176837, 'y': 261.359900373599}",Other Posterior Circulation
1,1.2.826.0.1.3680043.8.498.10022796280698534221...,1.2.826.0.1.3680043.8.498.53868409774237283281...,"{'x': 194.87253141831238, 'y': 178.32675044883...",Right Middle Cerebral Artery
2,1.2.826.0.1.3680043.8.498.10023411164590664678...,1.2.826.0.1.3680043.8.498.24186535344744886473...,"{'x': 189.23979878597123, 'y': 209.19184886465...",Right Middle Cerebral Artery
3,1.2.826.0.1.3680043.8.498.10030095840917973694...,1.2.826.0.1.3680043.8.498.75217084841854214544...,"{'x': 208.2805049088359, 'y': 229.78962131837307}",Right Infraclinoid Internal Carotid Artery
4,1.2.826.0.1.3680043.8.498.10034081836061566510...,1.2.826.0.1.3680043.8.498.71237104731452368587...,"{'x': 249.86745590416498, 'y': 220.623044646393}",Anterior Communicating Artery


In [10]:
# Input the series id and get the SOPINstanceUID and the coordinates as an output

#seriesid = series_with.iloc[1]
seriesid = '1.2.826.0.1.3680043.8.498.99892390884723813599532075083872271516'
localizer = train_localizers[train_localizers['SeriesInstanceUID'].eq(seriesid)][['SeriesInstanceUID','SOPInstanceUID', 'coordinates']]
print("SeriesInstanceUID: ", localizer['SeriesInstanceUID'])
print("SOPInstanceUID: ",localizer['SOPInstanceUID'])
print("Coordinates: ",localizer['coordinates'])

SeriesInstanceUID:  2281    1.2.826.0.1.3680043.8.498.99892390884723813599...
2282    1.2.826.0.1.3680043.8.498.99892390884723813599...
2283    1.2.826.0.1.3680043.8.498.99892390884723813599...
2284    1.2.826.0.1.3680043.8.498.99892390884723813599...
Name: SeriesInstanceUID, dtype: object
SOPInstanceUID:  2281    1.2.826.0.1.3680043.8.498.12398549862508001109...
2282    1.2.826.0.1.3680043.8.498.12398549862508001109...
2283    1.2.826.0.1.3680043.8.498.21598979799967012280...
2284    1.2.826.0.1.3680043.8.498.21598979799967012280...
Name: SOPInstanceUID, dtype: object
Coordinates:  2281    {'x': 139.11888111888112, 'y': 283.97202797202...
2282    {'x': 63.132075471698116, 'y': 195.8238993710692}
2283    {'x': 14.510769230769231, 'y': 247.53230769230...
2284    {'x': 167.3230769230769, 'y': 265.64923076923077}
Name: coordinates, dtype: object


In [13]:
# Trying to visualize which series have more than one aneurysm as their label

train_localizers.groupby("SeriesInstanceUID").count()

,SOPInstanceUID,coordinates,location
SeriesInstanceUID,,,
1.2.826.0.1.3680043.8.498.10005158603912009425635473100344077317,1,1,1
1.2.826.0.1.3680043.8.498.10022796280698534221758473208024838831,1,1,1
1.2.826.0.1.3680043.8.498.10023411164590664678534044036963716636,1,1,1
1.2.826.0.1.3680043.8.498.10030095840917973694487307992374923817,1,1,1
1.2.826.0.1.3680043.8.498.10034081836061566510187499603024895557,1,1,1
...,...,...,...
1.2.826.0.1.3680043.8.498.99800061424469215274400822361248888410,1,1,1
1.2.826.0.1.3680043.8.498.99804081131933373817667779922320327920,1,1,1
1.2.826.0.1.3680043.8.498.99887675554378211308175946117895608384,2,2,2
